# Preprocessing Dataset CWRU
Validasi awal, pelabelan metadata, resampling, pembersihan, dan penggabungan 40 rekaman bearing. Jalankan cell dari atas ke bawah. File `.mat` asli tidak diubah.

## 1. Library dan lokasi dataset
Notebook ini disimpan di folder `Tugas Besar`, sejajar dengan folder `cwru_download_package`. Jika library belum tersedia, jalankan `%pip install numpy pandas scipy`.

In [1]:
from pathlib import Path
import csv
import re
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import resample_poly

ROOT = Path.cwd()
DATA_DIR = ROOT / "cwru_download_package" / "CWRU_40_files"
OUTPUT_DIR = ROOT / "outputs"
TARGET_FS = 12_000
print("Folder dataset:", DATA_DIR)
print("Ditemukan:", DATA_DIR.exists())

Folder dataset: d:\python\sem 5\Pemrosesan sinyal\Tugas Besar\cwru_download_package\CWRU_40_files
Ditemukan: True


## 2. Metadata rekaman dan label
Ukuran cacat tidak berlaku untuk rekaman Normal (ditulis kosong dalam CSV). Label beban motor mengikuti urutan 0–3 HP dalam setiap kelompok. Verifikasi label ini dengan tabel resmi CWRU sebelum pemodelan.

In [2]:
GROUPS = {
    ('Normal', None): [97, 98, 99, 100],
    ('Inner Race', 0.007): [105, 106, 107, 108],
    ('Ball Fault', 0.007): [118, 119, 120, 121],
    ('Outer Race', 0.007): [130, 131, 132, 133],
    ('Inner Race', 0.014): [169, 170, 171, 172],
    ('Ball Fault', 0.014): [185, 186, 187, 188],
    ('Outer Race', 0.014): [197, 198, 199, 200],
    ('Inner Race', 0.021): [209, 210, 211, 212],
    ('Ball Fault', 0.021): [222, 223, 224, 225],
    ('Outer Race', 0.021): [234, 235, 236, 237],
}
LABELS = {}
for (condition, size), ids in GROUPS.items():
    for hp, file_id in enumerate(ids):
        LABELS[file_id] = (condition, size, hp)
assert len(LABELS) == 40


## 3. Fungsi pembaca sinyal Drive End
Variabel dipilih berdasarkan ID file, misalnya `99.mat` → `X099_DE_time`, sehingga keberadaan `X098_DE_time` tambahan tidak menyebabkan salah pilih.

In [3]:
def read_de(path, file_id):
    data = loadmat(path)
    expected = f'X{file_id:03d}_DE_time'
    if expected not in data:
        raise ValueError(f'{path.name}: variabel {expected} tidak ada; tersedia: '
                         f'{[k for k in data if k.endswith("_DE_time")]}')
    arr = np.asarray(data[expected])
    if not np.issubdtype(arr.dtype, np.number) or np.iscomplexobj(arr):
        raise ValueError(f'{path.name}: sinyal tidak berupa angka real')
    if arr.ndim > 2 or (arr.ndim == 2 and min(arr.shape) != 1):
        raise ValueError(f'{path.name}: dimensi sinyal tidak sesuai {arr.shape}')
    return arr.reshape(-1).astype(np.float64), expected


## 4. Validasi awal seluruh file
Periksa 40 ID, panjang sinyal, serta nilai `NaN`/`Inf`. Tidak ada perubahan data pada tahap ini.

In [4]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_DIR}")
by_id = {}
for path in DATA_DIR.glob("*.mat"):
    match = re.fullmatch(r"\s*(\d+)\s*", path.stem)
    if not match:
        raise ValueError(f"Nama file tidak dikenali: {path.name}")
    file_id = int(match.group(1))
    if file_id in by_id:
        raise ValueError(f"ID duplikat: {file_id}")
    by_id[file_id] = path
missing = sorted(set(LABELS) - set(by_id))
extra = sorted(set(by_id) - set(LABELS))
if missing or extra:
    raise ValueError(f"ID tidak cocok. Kurang: {missing}; tambahan: {extra}")

validation_rows = []
for file_id in sorted(LABELS):
    path = by_id[file_id]
    signal, key = read_de(path, file_id)
    invalid = int(np.count_nonzero(~np.isfinite(signal)))
    validation_rows.append({"source_file": path.name, "signal_key": key,
                            "total_samples": len(signal), "invalid_samples": invalid,
                            "valid_samples": len(signal) - invalid})
validation_df = pd.DataFrame(validation_rows)
print(f"File: {len(validation_df)} | Sampel: {validation_df.total_samples.sum():,} | Invalid: {validation_df.invalid_samples.sum():,}")
display(validation_df)

File: 40 | Sampel: 6,089,696 | Invalid: 0


,source_file,signal_key,total_samples,invalid_samples,valid_samples
0,97.mat,X097_DE_time,243938,0,243938
1,98 .mat,X098_DE_time,483903,0,483903
2,99.mat,X099_DE_time,485063,0,485063
3,100.mat,X100_DE_time,485643,0,485643
4,105.mat,X105_DE_time,121265,0,121265
5,106.mat,X106_DE_time,121991,0,121991
6,107.mat,X107_DE_time,122136,0,122136
7,108.mat,X108_DE_time,122917,0,122917
8,118.mat,X118_DE_time,122571,0,122571
9,119.mat,X119_DE_time,121410,0,121410


## 5. Resampling, cleaning, dan penggabungan
Rekaman Normal diasumsikan 48 kHz dan Fault 12 kHz sesuai kelompok data yang dipilih. Normal diturunkan ke 12 kHz dengan `resample_poly` (antialiasing). Jika terdapat `NaN`/`Inf`, proses dihentikan agar indeks waktu tidak rusak. Nilai amplitudo ekstrem **tidak** dihapus. CSV ditulis bertahap agar RAM tidak penuh.

In [5]:
def run_preprocessing():
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f'Folder dataset tidak ditemukan: {DATA_DIR}')
    paths = list(DATA_DIR.glob('*.mat'))
    by_id = {}
    for path in paths:
        # Mendukung nama seperti '98 .mat', tanpa mengubah file sumber.
        match = re.fullmatch(r'\s*(\d+)\s*', path.stem)
        if not match:
            raise ValueError(f'Nama file tidak dikenali: {path.name}')
        file_id = int(match.group(1))
        if file_id in by_id:
            raise ValueError(f'Duplikat ID file {file_id}: {by_id[file_id].name}, {path.name}')
        by_id[file_id] = path
    missing = sorted(set(LABELS) - set(by_id))
    extra = sorted(set(by_id) - set(LABELS))
    if missing or extra:
        raise ValueError(f'ID dataset tidak cocok. Kurang: {missing}; tambahan: {extra}')

    OUTPUT_DIR.mkdir(exist_ok=True)
    meta_rows, report_rows = [], []
    total_original = total_output = 0
    combined = OUTPUT_DIR / 'bearing_clean.csv'
    with combined.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(['source_file', 'condition', 'defect_size_inch', 'motor_load_hp',
                         'original_fs_hz', 'sampling_rate_hz', 'sample_index',
                         'time_seconds', 'vibration'])
        for number, file_id in enumerate(sorted(LABELS), start=1):
            path = by_id[file_id]
            condition, defect_size, hp = LABELS[file_id]
            original_fs = 48_000 if condition == 'Normal' else 12_000
            signal, key = read_de(path, file_id)
            invalid = int(np.count_nonzero(~np.isfinite(signal)))
            if invalid:
                # Jangan merapatkan waktu dengan menghapus sampel sebelum resampling.
                raise ValueError(f'{path.name}: ada {invalid} NaN/Inf. '
                                 'Perlu penanganan gap sinyal sebelum resampling.')
            if not len(signal):
                raise ValueError(f'{path.name}: sinyal kosong')
            clean = resample_poly(signal, up=1, down=4) if original_fs == 48_000 else signal
            if not np.all(np.isfinite(clean)):
                raise ValueError(f'{path.name}: hasil resampling tidak valid')
            size_out = '' if defect_size is None else defect_size
            # Penulisan bertahap menghindari pembuatan DataFrame jutaan baris di RAM.
            for start in range(0, len(clean), 50_000):
                end = min(start + 50_000, len(clean))
                writer.writerows((path.name, condition, size_out, hp, original_fs,
                                  TARGET_FS, i, f'{i/TARGET_FS:.9f}',
                                  f'{float(clean[i]):.12g}')
                                 for i in range(start, end))
            meta_rows.append([path.name, key, condition, size_out, hp,
                              original_fs, TARGET_FS, len(signal), len(clean)])
            report_rows.append([path.name, 'OK', len(signal), invalid, len(clean),
                                'Resampled 48k to 12k' if original_fs == 48_000
                                else 'No resampling'])
            total_original += len(signal)
            total_output += len(clean)
            print(f'[{number:02d}/40] {path.name:<9} {condition:<11} '
                  f'{hp} HP | {original_fs} -> {TARGET_FS} Hz | '
                  f'{len(signal):,} -> {len(clean):,} sampel', flush=True)

    with (OUTPUT_DIR / 'metadata_rekaman.csv').open('w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f)
        w.writerow(['source_file', 'signal_key', 'condition', 'defect_size_inch',
                    'motor_load_hp', 'original_fs_hz', 'sampling_rate_hz',
                    'original_samples', 'output_samples'])
        w.writerows(meta_rows)
    with (OUTPUT_DIR / 'preprocessing_report.csv').open('w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f)
        w.writerow(['source_file', 'status', 'original_samples', 'invalid_samples',
                    'output_samples', 'notes'])
        w.writerows(report_rows)
    print('\nSELESAI')
    print(f'Rekaman: {len(meta_rows)}')
    print(f'Sampel asli: {total_original:,}')
    print(f'Sampel hasil: {total_output:,}')
    print(f'Output: {OUTPUT_DIR}')


run_preprocessing()

[01/40] 97.mat    Normal      0 HP | 48000 -> 12000 Hz | 243,938 -> 60,985 sampel
[02/40] 98 .mat   Normal      1 HP | 48000 -> 12000 Hz | 483,903 -> 120,976 sampel
[03/40] 99.mat    Normal      2 HP | 48000 -> 12000 Hz | 485,063 -> 121,266 sampel
[04/40] 100.mat   Normal      3 HP | 48000 -> 12000 Hz | 485,643 -> 121,411 sampel
[05/40] 105.mat   Inner Race  0 HP | 12000 -> 12000 Hz | 121,265 -> 121,265 sampel
[06/40] 106.mat   Inner Race  1 HP | 12000 -> 12000 Hz | 121,991 -> 121,991 sampel
[07/40] 107.mat   Inner Race  2 HP | 12000 -> 12000 Hz | 122,136 -> 122,136 sampel
[08/40] 108.mat   Inner Race  3 HP | 12000 -> 12000 Hz | 122,917 -> 122,917 sampel
[09/40] 118.mat   Ball Fault  0 HP | 12000 -> 12000 Hz | 122,571 -> 122,571 sampel
[10/40] 119.mat   Ball Fault  1 HP | 12000 -> 12000 Hz | 121,410 -> 121,410 sampel
[11/40] 120.mat   Ball Fault  2 HP | 12000 -> 12000 Hz | 121,556 -> 121,556 sampel
[12/40] 121.mat   Ball Fault  3 HP | 12000 -> 12000 Hz | 121,556 -> 121,556 sampel
[13/4

## 6. Memeriksa hasil preprocessing
Tampilkan metadata dan laporan. File `bearing_clean.csv` dapat berukuran besar; tampilkan hanya lima baris awal.

In [6]:
metadata_df = pd.read_csv(OUTPUT_DIR / "metadata_rekaman.csv")
report_df = pd.read_csv(OUTPUT_DIR / "preprocessing_report.csv")
print("Jumlah rekaman:", len(metadata_df))
print("Sampel keluaran:", int(report_df.output_samples.sum()))
print("Sampel invalid:", int(report_df.invalid_samples.sum()))
display(metadata_df.head(10))
display(report_df.groupby("notes", dropna=False)[["original_samples", "output_samples"]].sum())
display(pd.read_csv(OUTPUT_DIR / "bearing_clean.csv", nrows=5))

Jumlah rekaman: 40
Sampel keluaran: 4815787
Sampel invalid: 0


,source_file,signal_key,condition,defect_size_inch,motor_load_hp,original_fs_hz,sampling_rate_hz,original_samples,output_samples
0,97.mat,X097_DE_time,Normal,NaN,0,48000,12000,243938,60985
1,98 .mat,X098_DE_time,Normal,NaN,1,48000,12000,483903,120976
2,99.mat,X099_DE_time,Normal,NaN,2,48000,12000,485063,121266
3,100.mat,X100_DE_time,Normal,NaN,3,48000,12000,485643,121411
4,105.mat,X105_DE_time,Inner Race,0.007,0,12000,12000,121265,121265
5,106.mat,X106_DE_time,Inner Race,0.007,1,12000,12000,121991,121991
6,107.mat,X107_DE_time,Inner Race,0.007,2,12000,12000,122136,122136
7,108.mat,X108_DE_time,Inner Race,0.007,3,12000,12000,122917,122917
8,118.mat,X118_DE_time,Ball Fault,0.007,0,12000,12000,122571,122571
9,119.mat,X119_DE_time,Ball Fault,0.007,1,12000,12000,121410,121410


,original_samples,output_samples
notes,,
No resampling,4391149,4391149
Resampled 48k to 12k,1698547,424638


,source_file,condition,defect_size_inch,motor_load_hp,original_fs_hz,sampling_rate_hz,sample_index,time_seconds,vibration
0,97.mat,Normal,NaN,0,48000,12000,0,0.000000,0.055947
1,97.mat,Normal,NaN,0,48000,12000,1,0.000083,0.015231
2,97.mat,Normal,NaN,0,48000,12000,2,0.000167,-0.079317
3,97.mat,Normal,NaN,0,48000,12000,3,0.000250,0.045434
4,97.mat,Normal,NaN,0,48000,12000,4,0.000333,0.047412


## 7. Output dan catatan metodologi
- `outputs/bearing_clean.csv`: sampel sinyal, ID sumber, label, indeks dan waktu.
- `outputs/metadata_rekaman.csv`: ringkasan setiap rekaman.
- `outputs/preprocessing_report.csv`: catatan cleaning dan resampling.

**Catatan penelitian:** segmen dari rekaman sumber yang sama tidak boleh tersebar secara acak antara train dan test jika ingin mengukur generalisasi ke rekaman baru. Jumlah file lengkap tidak otomatis menjamin evaluasi bebas kebocoran data.